# Episode 2: Bronze tables

Every folder in the landing volumes becomes one **Delta table** in bronze:

```
/Volumes/<project>_sources/landing/<system>/<folder>   ->   <project>_dev.bronze_<system>.<folder>
```

Bronze = the data **as delivered**: no cleaning, no deduplication, no type fixes. We only add two technical columns:
- `_source_file` - which file the row came from,
- `_ingested_at` - when we loaded it.

`load_date` is added by Spark on its own: it recognises the `load_date=YYYY-MM-DD` folders as a partition column.

This is a **full reload** (`mode("overwrite")`): every run reads all files again and replaces the tables, so re-running is safe.
Incremental loading (only new files) comes later with Auto Loader.

In [0]:
CATALOG_PREFIX = "moustache_markt"
ENV = "dev"

SOURCES = f"{CATALOG_PREFIX}_sources"   # files from the source systems (landing volumes)
TARGET = f"{CATALOG_PREFIX}_{ENV}"     # our platform: bronze_<system> schemas live here

In [0]:
# The same readers as in 01_reading_files. A path can be one file or a whole folder:
# for a folder Spark reads every file inside it, including the load_date=... subfolders.


def read_csv(path):
    # POS: "|" as the delimiter, Windows encoding (cp1252), first line = column names
    return spark.read.option("header", True).option("sep", "|").option("encoding", "cp1252").csv(path)


def read_json(path):
    # e-commerce: the whole file is one JSON array, so Spark has to read it as a single document
    return spark.read.option("multiLine", True).json(path)


def read_jsonl(path):
    # CRM: one JSON object per line - the default for spark.read.json
    return spark.read.json(path)


def read_parquet(path):
    # ERP: the schema is stored inside the file, no options needed
    return spark.read.parquet(path)


READERS = {"csv": read_csv, "json": read_json, "jsonl": read_jsonl, "parquet": read_parquet}

## Load every landing folder into bronze

For each folder we check the extension of its first file (like `read_all_files()` in episode 1),
read the **whole folder** with the matching reader and save it as a table.

In [0]:
from pyspark.sql import functions as F


def load_bronze_table(folder_path, table_name):
    # folder -> first load_date=... subfolder -> first file -> its extension picks the reader
    first_day = dbutils.fs.ls(folder_path)[0].path
    sample = dbutils.fs.ls(first_day)[0].path
    extension = sample.split(".")[-1]

    df = (
        READERS[extension](folder_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))   # hidden column every file reader has
        .withColumn("_ingested_at", F.current_timestamp())
    )
    # overwriteSchema: if the source schema changed, replace the table schema instead of failing
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(table_name)


# Landing volumes = one volume per source system (erp, crm, ecom, pos)
volumes = spark.sql(
    """
    SELECT volume_schema, volume_name
    FROM system.information_schema.volumes
    WHERE volume_catalog = :catalog
    ORDER BY volume_name
    """,
    args={"catalog": SOURCES},
).collect()

for volume in volumes:
    volume_path = f"/Volumes/{SOURCES}/{volume.volume_schema}/{volume.volume_name}"
    for folder in dbutils.fs.ls(volume_path):
        table = folder.name.rstrip("/")
        table_name = f"{TARGET}.bronze_{volume.volume_name}.{table}"

        load_bronze_table(f"{volume_path}/{table}", table_name)
        print(f"{table_name:<50} {spark.table(table_name).count():>12,} rows")

## Check what landed in bronze

In [0]:
display(spark.sql(
    """
    SELECT table_schema, table_name, table_type, created
    FROM system.information_schema.tables
    WHERE table_catalog = :catalog AND startswith(table_schema, 'bronze_')
    ORDER BY table_schema, table_name
    """,
    args={"catalog": TARGET},
))

In [0]:
# Example: POS lines - every column is a string, because CSV has no types (we did not ask Spark to guess them)
display(spark.table(f"{TARGET}.bronze_pos.receipt_line").limit(10))